In [2]:
import torch
import torch.nn as nn
import numpy as np

#B -> batch ,  E -> input dimension, H -> hidden  dimension,  T -> series stpe
B, E, H = 1, 128, 3

In [3]:
def prepare_inputs():
    np.random.seed(42)

    vocab = {"播放": 0, "周杰伦": 1, "的": 2, "《稻香》": 3}
    tokens = ["播放", "周杰伦", "的", "《稻香》"]
    ids = [vocab[t] for t in tokens]

    V = len(vocab)
    em_table = np.random.rand(V, E).astype(np.float32)

    x_np = em_table[ids][None]
    return tokens, x_np

In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [5]:
def lstm_numpy(x_np, weights):
    # U_f forget gate, U_i input gate, U_c candidate,  U_o output
    U_f, W_f, U_i, W_i, U_c, W_c, U_o, W_o = weights
    B_local, T_local, _ = x_np.shape
    h_prev = np.zeros((B_local, H), dtype=np.float32)
    c_prev = np.zeros((B_local, H), dtype=np.float32)

    steps = []
    for t in range(T_local):
        x_t = x_np[:, t, :]

        # forget gate
        f_t = sigmoid(x_t @ U_f + h_prev @ W_f)

        # input and candidate memory
        i_t = sigmoid(x_t @ U_i + h_prev @ W_i)
        c_tilde_t = np.tanh(x_t @ U_c + h_prev @ W_c)

        # cell update state
        c_t = f_t * c_prev + i_t * c_tilde_t

        # output and hidden state
        o_t = sigmoid(x_t @ U_o + h_prev @ W_o)
        h_t = o_t * np.tanh(c_t)

        steps.append(h_t)
        h_prev, c_prev = h_t, c_t
    outputs = np.stack(steps, axis=1)
    return outputs, h_prev, c_prev



In [6]:
_, x_np = prepare_inputs()
np.random.seed(7)
U_f, W_f = np.random.randn(E, H).astype(np.float32), np.random.randn(H, H).astype(np.float32)
U_i, W_i = np.random.randn(E, H).astype(np.float32), np.random.randn(H, H).astype(np.float32)
U_c, W_c = np.random.randn(E, H).astype(np.float32), np.random.randn(H, H).astype(np.float32)
U_o, W_o = np.random.randn(E, H).astype(np.float32), np.random.randn(H, H).astype(np.float32)

weights_np = (U_f, W_f, U_i, W_i, U_c, W_c, U_o, W_o)

In [7]:
out_manual_np, hT_manual_np, cT_manual_np = lstm_numpy(x_np, weights_np)


In [8]:
out_manual_np.shape

(1, 4, 3)

In [9]:
hT_manual_np.shape

(1, 3)

In [10]:
cT_manual_np.shape

(1, 3)

In [11]:
out_manual_np

array([[[ 2.7683512e-03, -1.2375321e-07,  3.8353039e-05],
        [ 7.4904788e-01, -1.8654716e-06,  6.6107100e-07],
        [ 1.2572932e-02, -3.8905484e-05,  1.7104972e-05],
        [ 6.3237804e-01, -7.2920869e-09,  3.0335900e-04]]], dtype=float32)

In [12]:
hT_manual_np

array([[ 6.3237804e-01, -7.2920869e-09,  3.0335900e-04]], dtype=float32)

In [13]:
cT_manual_np

array([[ 0.7453765 , -0.01359617,  0.021516  ]], dtype=float32)